# XGLUE - Tuning Analysis
Review saved threshold-tuning output and run a full-dataset threshold sweep when spectral features are available.


In [ ]:
from pathlib import Path
import sys


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "spectral_code").exists() and (candidate / "pipelines").exists():
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from IPython.display import display

from spectral_code.evaluation.notebook_helpers import (
    artifact_status_dataframe,
    configure_notebook_style,
    dataset_overview_rows,
    display_clone_nonclone_pair_inspection,
    display_code_graph_side_by_side_examples,
    display_dataset_overview,
    display_global_threshold_tuning_summary,
    display_pipeline_validation,
    display_similarity_distribution_report,
    display_statistical_distribution_plots,
    display_tuning_report,
    graph_manifest_summary_dataframe,
    load_pairs_for_spec,
    load_tuning_results,
    notebook_output_dir,
    pair_stats_dataframe,
    plot_graph_coverage_from_timing,
    plot_graph_manifest_summary,
    plot_timing_stats,
    timing_stats_dataframe,
    xglue_spec,
)

configure_notebook_style()
spec = xglue_spec()
GRAPH_TYPES = ["ast", "cfg", "ddg", "pdg", "cpg"]
ARTIFACT_DIR = notebook_output_dir(spec)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {spec.data_dir}")
print(f"Output root: {spec.output_root}")
print(f"Notebook artifacts: {ARTIFACT_DIR}")


## Saved Tuning Results


In [ ]:
tuning_df = display_tuning_report(spec, include_pair_examples=True)
tuning_df.head() if not tuning_df.empty else tuning_df


## Global Threshold Sweep


In [ ]:
if spec.features_manifest.exists():
    sweep_df, best = display_global_threshold_tuning_summary(
        spec,
        graph_types=GRAPH_TYPES,
        optimize_for="f1",
        seed=42,
    )
    display(sweep_df.head() if not sweep_df.empty else sweep_df)
    best
else:
    print(f"Run 03_extract_spectral_features.py first. Missing: {spec.features_manifest}")


## Tuning Files


In [ ]:
for path in sorted(spec.output_root.glob("trained_*.json")):
    print(path)
